# 19 · Concurrency

*Beyond the contest* · 6 exercises

Do several things at once safely: start work and wait for all of it, pass values between workers, cap the number of workers, protect shared state, and stop waiting when a result is late.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-19) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-19)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import queue
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, TimeoutError

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 19.1 · Start workers and wait for all  `Core`

**Task.** Write `parallel_squares(nums)` that starts one `threading.Thread` per number, each storing its number's square in its own slot of a results list, then `join`s every thread before returning the list.

**Example.** `[1, 2, 3]` -> `[1, 4, 9]`

**Big O.** O(N) work spread over N threads; each writes only its own slot, so no lock is needed.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-1)

In [ ]:
# 19.1 Start workers and wait for all
def parallel_squares(nums):
    raise NotImplementedError

In [ ]:
def test_ex_19_1():
    assert parallel_squares([1, 2, 3]) == [1, 4, 9]
    assert parallel_squares([]) == []
    assert parallel_squares(list(range(50))) == [i * i for i in range(50)]


test_ex_19_1()
print('19.1 passed')

### 19.2 · A producer and a consumer  `Core`

**Task.** Write `pipeline_total(nums)` with two threads joined by a `queue.Queue`: a producer thread puts each number and then `None` to say it is finished, while a consumer thread takes items until it sees `None`, adding them up. Return the consumer's total after joining both.

**Example.** `[4, 5, 6]` -> `15`

**Big O.** O(N): every value passes through the queue once; the queue does the locking and the waiting.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-2)

In [ ]:
# 19.2 A producer and a consumer
def pipeline_total(nums):
    raise NotImplementedError

In [ ]:
def test_ex_19_2():
    assert pipeline_total([4, 5, 6]) == 15
    assert pipeline_total([]) == 0
    assert pipeline_total(list(range(10000))) == 49995000


test_ex_19_2()
print('19.2 passed')

### 19.3 · A pool with a fixed number of workers  `Practice`

**Task.** Write `pool_map(fn, items, workers)` that applies `fn` to every item using at most `workers` threads at a time and returns the results in input order. Use a `ThreadPoolExecutor(max_workers=workers)` in a `with` block and its `map`.

**Example.** `lambda x: x * 10, [1, 2, 3], 2` -> `[10, 20, 30]`

**Big O.** About ceil(N / W) rounds of work for W workers; the results keep the input order however the work finishes.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-3)

In [ ]:
# 19.3 A pool with a fixed number of workers
def pool_map(fn, items, workers):
    raise NotImplementedError

In [ ]:
def test_ex_19_3():
    assert pool_map(lambda x: x * 10, [1, 2, 3], 2) == [10, 20, 30]
    assert pool_map(str, [], 3) == []
    lock, active, peak = (threading.Lock(), [0], [0])

    def slow(x):
        with lock:
            active[0] += 1
            peak[0] = max(peak[0], active[0])
        time.sleep(0.01)
        with lock:
            active[0] -= 1
        return -x
    assert pool_map(slow, list(range(12)), 3) == [-x for x in range(12)]
    assert 1 <= peak[0] <= 3


test_ex_19_3()
print('19.3 passed')

### 19.4 · A counter protected by a lock  `Core`

**Task.** Write `locked_count(threads, increments)` that starts `threads` threads, each adding 1 to a shared counter `increments` times, and returns the final count. Guard every update with one `threading.Lock` (`with lock:`), because `count += 1` is a read, an add and a write that two threads can interleave.

**Example.** `4, 1000` -> `4000`

**Big O.** O(T * K) increments; the lock makes each read-modify-write happen as one step, so none is lost.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-4)

In [ ]:
# 19.4 A counter protected by a lock
def locked_count(threads, increments):
    raise NotImplementedError

In [ ]:
def test_ex_19_4():
    assert locked_count(4, 1000) == 4000
    assert locked_count(1, 0) == 0
    assert locked_count(8, 2500) == 20000


test_ex_19_4()
print('19.4 passed')

### 19.5 · Stop waiting after a timeout  `Practice`

**Task.** Write `result_within(task, seconds)` that runs `task()` on a worker thread and returns `(True, result)` if it finishes within `seconds`, or `(False, None)` otherwise, without waiting for a slow task to end. Submit it to a `ThreadPoolExecutor`, call `future.result(timeout=seconds)`, and `shutdown(wait=False)`.

**Example.** `lambda: 42, 1.0` -> `(True, 42)`; a task that sleeps 0.3 s with `0.05` -> `(False, None)`

**Big O.** Waits at most the timeout, however long the task takes.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-5)

In [ ]:
# 19.5 Stop waiting after a timeout
def result_within(task, seconds):
    raise NotImplementedError

In [ ]:
def test_ex_19_5():
    assert result_within(lambda: 42, 1.0) == (True, 42)
    start = time.monotonic()
    assert result_within(lambda: time.sleep(0.3), 0.05) == (False, None)
    assert time.monotonic() - start < 0.25


test_ex_19_5()
print('19.5 passed')

### 19.6 · Split a sum across workers  `Practice`

**Task.** Write `parallel_sum(nums, workers)` that cuts `nums` into `workers` contiguous chunks of near-equal size, sums each chunk on its own thread (a `ThreadPoolExecutor`), and adds the partial sums.

**Example.** `list(range(1, 101)), 4` -> `5050`

**Big O.** O(N / W) per worker for W workers, then O(W) to combine; each worker owns one slot of the partial results.

**Note.** In CPython, threads take turns running Python code (the GIL), so this shows the pattern rather than a speed-up; use processes for CPU-heavy work.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-19-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-19-6)

In [ ]:
# 19.6 Split a sum across workers
def parallel_sum(nums, workers):
    raise NotImplementedError

In [ ]:
def test_ex_19_6():
    assert parallel_sum(list(range(1, 101)), 4) == 5050
    assert parallel_sum([], 3) == 0
    assert parallel_sum([5], 8) == 5
    assert parallel_sum(list(range(1000)), 7) == 499500


test_ex_19_6()
print('19.6 passed')